# LLM재킹 탐지 규칙 시뮬레이션

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/llmjacking_detection.ipynb)

가상 조직의 LLM 사용량에 노출된 키의 비정상 호출을 섞고, 탐지 규칙과 차단 정책마다 탐지 지연, 요금, 정상 로그의 오탐을 비교합니다. 모든 로그는 합성이며 실제 공격 도구를 사용하거나 LLM·클라우드 서비스 API를 호출하지 않습니다.

## 실행 설정

위에서부터 순서대로 실행하세요. 기본값은 14일·50개 시드를 쓰는 전체 실험입니다. `QUICK = True`로 바꾸면 7일·3개 시드만 실행하므로 R4가 비교할 2주차 이력이 없습니다. `CHART_LANG`은 `ko`, `en`, `both` 중에서 고르고 `KRW_PER_USD`에서 가정 환율을 바꿀 수 있습니다.

실험의 토큰 단가는 고정 가정입니다. [원본 실험의 단가 참고 자료 · Claude API 요금](https://platform.claude.com/docs/en/about-claude/pricing)

In [ ]:
import json
import math
import os
import tempfile
import time
import unicodedata
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlopen

import numpy as np

QUICK = False
CHART_LANG = os.environ.get("CHART_LANG", "ko")
KRW_PER_USD = 1400

if CHART_LANG not in ("ko", "en", "both"):
    raise ValueError("CHART_LANG must be ko, en or both")
if not math.isfinite(KRW_PER_USD) or KRW_PER_USD <= 0:
    raise ValueError("KRW_PER_USD must be finite and positive")

VERSION = "1.1.0"
DAY, HOUR = 86400, 3600
DAYS, QUICK_DAYS = 14, 7
SEEDS, QUICK_SEEDS = 50, 3
START_DAY, QUICK_START_DAY = 11, 5
RATES = (0.02, 0.03, 0.04, 0.05, 0.08, 0.12, 0.20, 0.50)
KEYS = tuple(f"key-{i:02d}" for i in range(1, len(RATES) + 2))
# One integer denotes one fictional /24 (e.g. 3 = net-003).
DEV_NET, OPS_NET, ATTACK_NET, FIRST_DYNAMIC_NET = range(4)
INVOKE, LIST, ENABLE, DELETE = range(4)
OPS = ("Invoke", "ListModels", "EnableService", "DeleteLogging")
OK, THROTTLED, VALIDATION, MISSING = range(4)
RESULTS = ("ok", "throttled", "validation_error", "not_found")
LOW, MID, HIGH = range(3)
MODELS = (
    "low",
    "mid",
    "high",
    "absent-a",
    "absent-b",
    "absent-c",
)
# Fixed scenario prices in USD per million tokens.
PRICES = ((0.10, 0.50), (2.0, 10.0), (8.0, 40.0))
MONEY_UNIT = 10_000_000  # exact integer units per USD
PRICE_UNITS = (
    (np.array(PRICES) * MONEY_UNIT / 1_000_000)
    .round()
    .astype("i8")
)
PROFILES = {"short": (1500, 600), "long": (60000, 2000)}
SCENARIOS = ("base", "enable", "delete")
DEV_KEYS = 3
WORK = (9, 18)
OFF_HOURS, WEEKEND = 0.20, 0.35
DEPLOY_P, DEPLOY_MULT, DEPLOY_SECONDS = 0.2, (2.0, 3.0), 1800
WEEK = 7 * DAY
OPS_ROTATE_P, DEV_ROTATE_P = 0.3, 0.5
MODEL_TRIAL_P, MODEL_TRIAL_CALLS, MODEL_TRIAL_SECONDS = (
    0.4,
    (1, 20),
    600,
)
HEALTH_KEY, SMOKE_KEY = DEV_KEYS, DEV_KEYS + 1
HEALTH_INTERVAL, SMOKE_SECONDS, SMOKE_CALLS = 60, 600, (5, 15)
TYPO_P, TYPO_CALLS = 0.1, (5, 30)
ENABLE_P = 1 / 90
SECONDARY_P = 0.15
CAPS = (1024, 2048, 4096, 8192)
OUT_MEDIAN, OUT_SIGMA, NORMAL_INPUT = 400, 0.75, 1000
NORMAL_ERROR = (
    0.001,
    0.0002,
    0.0002,
)  # throttle, validation, missing
PROBE_SECONDS, PAUSE, AMP_SECONDS, RAMP = 7200, 1800, 28800, 600
PEAK, PROBES, SPRAY, THROTTLE_RANGE = (
    1100,
    (20, 60),
    (3, 8),
    (0.05, 0.15),
)
ROWS = np.dtype(
    [
        ("second", "i4"),
        ("key", "u1"),
        ("source", "i4"),
        ("op", "u1"),
        ("model", "u1"),
        ("result", "u1"),
        ("count", "i4"),
        ("max_tokens", "i4"),
        ("input_tokens", "i8"),
        ("output_tokens", "i8"),
    ]
)

ORIGINAL_RULES = tuple(f"R{i}" for i in range(1, 8))
RULES = (*ORIGINAL_RULES, "R1b", "R3b")
PROBE_WINDOW, PROBE_N = 600, 5
MODEL_HISTORY, LEARN = 7 * DAY, 3 * DAY
RATE_WINDOW, RATE_K, RATE_MIN = 300, 10, 300
BURN_WINDOW, BURN_HISTORY, BURN_K, BURN_Q = (
    HOUR,
    7 * 24,
    5,
    0.99,
)
BURN_MIN_HOURS = 24
ERROR_WINDOW, ERROR_N = 600, 10
PRELOAD_NETWORK = False
PROBE_RATIO_K, PROBE_HISTORY = 3, 7 * DAY
CORRELATION_WINDOW = 600

P5_RULES = ("R1b", "R3b", "R6", "R7", "R4", "R5")
POLICIES = {
    "P1": ("R4",),
    "P2": ("R1", "R6", "R7"),
    "P3": ORIGINAL_RULES,
    "P4": ORIGINAL_RULES,
    "P5": P5_RULES,
    "P6": P5_RULES,
}
BLOCKING = ("P4", "P6")
METRICS = (*RULES, *POLICIES)

LINE_WIDTH = 64

if QUICK:
    days, trials, start = (
        QUICK_DAYS,
        QUICK_SEEDS,
        QUICK_START_DAY,
    )
else:
    days, trials, start = DAYS, SEEDS, START_DAY
fx = KRW_PER_USD

## 실행 머리말

UTC 실행 시각과 시뮬레이션 버전을 기록하고 트래픽, 요금, 규칙, 차단 정책에 적용할 가정을 출력합니다.

In [ ]:
def width(text):
    return sum(
        2 if unicodedata.east_asian_width(c) in "WF" else 1
        for c in text
    )


def line(text=""):
    while width(text) > LINE_WIDTH:
        end = next(
            i
            for i in range(len(text))
            if width(text[: i + 1]) > LINE_WIDTH
        )
        space = text.rfind(" ", 0, end + 1)
        cut = space if space > end // 2 else end
        print(text[:cut])
        text = text[cut:].lstrip()
    print(text)


def assumptions(days, seeds, start_day, fx):
    return [
        f"기간 {days}일 / 시드 0~{seeds - 1} / "
        f"공격 {start_day}일째 UTC 균등 시각",
        f"기본 {DAYS}일·{SEEDS}시드·{START_DAY}일째 / quick "
        f"{QUICK_DAYS}일·{QUICK_SEEDS}시드·{QUICK_START_DAY}일째",
        f"정상 키 {len(RATES)}개 / 개발 {DEV_KEYS}, "
        f"운영 {len(RATES) - DEV_KEYS}",
        "키별 평균 호출/초 " + " ".join(map(str, RATES)),
        f"월요일 00시 시작 / 평일·주말 {WORK[0]}~{WORK[1]}시 가중치 1",
        f"그 외 시간 {OFF_HOURS}배 / 주말 추가 {WEEKEND}배 / 주평균 정규화",
        f"배포 확률 {DEPLOY_P}/일 / {DEPLOY_SECONDS // 60}분 / "
        f"조직 전체 {DEPLOY_MULT}배 균등",
        "기본 트래픽은 키·초별 포아송, 같은 행의 호출은 속성 공유",
        f"개발 low, 운영 mid / 짝수 번호 키 보조 모델 확률 {SECONDARY_P}",
        f"정상 입력 {NORMAL_INPUT} / 출력 로그정규 중앙 "
        f"{OUT_MEDIAN}, σ {OUT_SIGMA}",
        "일반 호출 max_tokens 균등 " + "/".join(map(str, CAPS)),
        "출력은 1 이상, max_tokens 이하로 반올림·절단",
        "정상 throttle/validation/missing 확률 "
        + "/".join(map(str, NORMAL_ERROR)),
        f"가상 /24 개발 net-{DEV_NET:03}, 운영 net-{OPS_NET:03}, "
        f"공격 net-{ATTACK_NET:03}",
        f"운영 키마다 일 {OPS_ROTATE_P}, "
        f"개발 키마다 주 {DEV_ROTATE_P} 확률로 새 대역",
        "전환 시각 균등 / 조직 내 미사용 대역으로 이후 호출 이동",
        f"개발 키마다 주 {MODEL_TRIAL_P}로 미사용 모델 시도 / "
        "high 포함, 후보 소진 시 종료",
        f"모델 시도 {MODEL_TRIAL_CALLS}회 / "
        f"임의 {MODEL_TRIAL_SECONDS // 60}분 내 균등",
        f"{KEYS[HEALTH_KEY]} {HEALTH_INTERVAL}초마다 1토큰 헬스체크",
        f"{KEYS[SMOKE_KEY]} 배포 시작 후 {SMOKE_SECONDS // 60}분에 "
        f"1토큰 스모크 {SMOKE_CALLS}회",
        f"같은 배포 창에 확률 {TYPO_P}로 모델 오타 not_found {TYPO_CALLS}회",
        f"정상 조직 EnableService 확률 1/{1 / ENABLE_P:g}·일, "
        f"{KEYS[SMOKE_KEY]} / DeleteLogging 없음",
        f"탐색 {PROBE_SECONDS // 60}분 / "
        f"단일 토큰 low {PROBES}건 / 목록 조회 1회",
        "첫 탐색은 시작 초 / 나머지는 중복 없는 균등 시각",
        f"없는 모델 {len(MODELS) - 3}종 균등 / 오류 {SPRAY}건 / "
        f"오류 상한 {CAPS[0]}",
        f"정지 {PAUSE // 60}분 / 증폭 {AMP_SECONDS // 3600}시간 / "
        f"선형 상승 {RAMP}초",
        f"증폭 평균 최고 {PEAK}/초 / high / 상한 {CAPS[-1]}",
        f"throttled 확률 {THROTTLE_RANGE} 균등 추출, 초별 이항 분할",
        "기본·삭제는 정상 키 균등 추출 / 삭제는 증폭 1초 전",
        "활성화는 조직 내 미사용 계정 key-09, 1초 뒤 증폭",
        "관리·목록·실패 무료 / 성공 Invoke만 입력·출력 과금",
        "단가 USD/MTok 입력·출력 "
        + " ".join(
            f"{m} {a:g}/{b:g}"
            for m, (a, b) in zip(MODELS, PRICES)
        ),
        "프로필 입력/출력 "
        + " ".join(
            f"{p} {i}/{o}" for p, (i, o) in PROFILES.items()
        ),
        f"탐색 성공 출력 1 / {MONEY_UNIT:,} 정수 단위 = 1 USD",
        f"환율 1 USD = {fx:g} KRW",
        f"R1 {PROBE_WINDOW // 60}분 내 상한 ≤1 호출 {PROBE_N}건 이상",
        f"R2 학습 {LEARN // DAY}일 / "
        f"직전 {MODEL_HISTORY // DAY}일 시도 이력",
        f"R1b R1과 같은 최소 건수 / {LEARN // DAY}일 학습 후 "
        f"현재 창 비율 > 과거 비율 ×{PROBE_RATIO_K}",
        f"현재 창 직전 최대 {PROBE_HISTORY // DAY}일 Invoke 기준 / "
        "과거 호출 0이면 보류",
        "R1b 재경보 / 일반 호출이 줄면 고정 헬스체크 비율 상승",
        f"R3 첫 {LEARN // DAY}일 관찰 학습, 이후 최초 /24 / "
        f"사전 등록 {PRELOAD_NETWORK}",
        f"R3b 같은 키의 R3와 R1b 또는 R2가 {CORRELATION_WINDOW}초 "
        "미만 간격, 나중 신호 초 경보",
        f"R4 {RATE_WINDOW // 60}분 / 과거 동일 요일·시 5분 구간 중앙값",
        f"R4 배수 {RATE_K}, 최소 {RATE_MIN}건 / 이력 없는 시간 제외",
        f"R5 {BURN_WINDOW // 60}분 / "
        f"직전 최대 {BURN_HISTORY}개 완료 시간",
        f"R5 최소 이력 {BURN_MIN_HOURS}시간 / "
        f"p{100 * BURN_Q:g}의 {BURN_K}배 초과",
        f"R6 활성화·삭제 즉시 / "
        f"R7 {ERROR_WINDOW // 60}분 오류 {ERROR_N}건",
        "윈도는 (t−폭,t] / 초 단위 동시 처리 / 초과 구간당 1회",
        "P1=R4 / P2=R1|R6|R7 / P3=R1~R7 OR / P4=P3 후 키 차단",
        "P5=" + "|".join(P5_RULES) + " / P6=P5 후 키 차단",
        "차단은 경보 다음 초부터 실험 종료까지 / 정상 시점부터 적용",
        "R5 차단 키는 그 초 최대 청구 키, 동률이면 작은 번호",
        "탐지는 공격 키·행위 또는 해당 집계창에 공격 기여 필요",
        "공격 종료까지 평가 / 미탐은 지연·경보 비용 집계에서 제외",
        "경보 전 호출 수·비용은 경보 초 포함 / 호출 수는 실패 포함 Invoke",
        "P4·P6 사전 차단 별도 기록 / 오탐 분모는 학습 포함 전체 조직·일",
        "정책 OR은 같은 키·초 1건 / 차단 정책은 실제 차단당 1건",
        "시나리오끼리 정상 로그·증폭 난수 공유 / 두 프로필 별도 판정",
    ]


utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
setup = assumptions(days, trials, start, fx)
line(f"UTC {utc}   |   v{VERSION}")
line("전체 로그는 합성이며 실제 고객 데이터는 없습니다.")
for item in setup:
    line(item)
line(f"차트 언어 {CHART_LANG} / PNG 150 dpi")

## 정상 조직과 공격 시나리오 생성

정상 로그에는 요일·시간대별 호출, 배포, 헬스체크, 모델 시도, 네트워크 변경을 반영합니다. 공격 로그는 `base`(탐색 후 증폭), `enable`(서비스 활성화 후 증폭), `delete`(기록 삭제 뒤 증폭) 세 조건의 합성 이벤트로 만듭니다. 짧은 입력과 긴 입력에서 각각 요금을 계산합니다.

In [ ]:
def empty_rows(n=0):
    return np.zeros(n, dtype=ROWS)


def ordered(*parts):
    rows = np.concatenate(parts)
    return rows[np.argsort(rows["second"], kind="stable")]


def normal_calls(seconds, model, cap, rng):
    rows = empty_rows(len(seconds))
    rows["second"], rows["count"], rows["model"] = (
        seconds,
        1,
        model,
    )
    rows["input_tokens"], rows["max_tokens"] = NORMAL_INPUT, cap
    out = rng.lognormal(
        np.log(OUT_MEDIAN), OUT_SIGMA, len(rows)
    )
    rows["output_tokens"] = np.clip(np.rint(out), 1, cap)
    return rows


def burst_times(rng, start, width, bounds):
    count = rng.integers(bounds[0], bounds[1] + 1)
    return start + rng.integers(width, size=count)


def normal_traffic(seed, days):
    """Aggregate traffic and mark sparse operational events."""
    rng = np.random.default_rng(
        np.random.SeedSequence([seed, 0])
    )
    ops_rng = np.random.default_rng(
        np.random.SeedSequence([seed, 2])
    )
    t = np.arange(days * DAY, dtype="i4")
    hour = t // HOUR % 24
    cycle = np.where(
        (hour >= WORK[0]) & (hour < WORK[1]), 1.0, OFF_HOURS
    )
    cycle *= np.where(t // DAY % 7 < 5, 1.0, WEEKEND)
    cycle /= (
        cycle.mean()
    )  # requested rates averaged over a complete week
    deployments = []
    enables = []
    for day in range(days):
        if rng.random() < DEPLOY_P:
            start = day * DAY + rng.integers(
                DAY - DEPLOY_SECONDS + 1
            )
            deployments.append(start)
            cycle[start : start + DEPLOY_SECONDS] *= (
                rng.uniform(*DEPLOY_MULT)
            )
        if ops_rng.random() < ENABLE_P:
            enables.append(day * DAY + ops_rng.integers(DAY))
    next_network = FIRST_DYNAMIC_NET
    streams = []
    for key, rate in enumerate(RATES):
        counts = rng.poisson(rate * cycle)
        sec = np.flatnonzero(counts)
        n = len(sec)
        rows = empty_rows(n)
        rows["second"] = sec
        rows["count"] = counts[sec]
        primary = LOW if key < DEV_KEYS else MID
        rows["model"] = primary
        if key % 2:
            rows["model"][rng.random(n) < SECONDARY_P] = (
                1 - primary
            )
        rows["max_tokens"] = rng.choice(CAPS, n)
        out = np.maximum(
            1,
            np.rint(
                rng.lognormal(np.log(OUT_MEDIAN), OUT_SIGMA, n)
            ),
        )
        rows["output_tokens"] = (
            np.minimum(out, rows["max_tokens"]) * rows["count"]
        )
        rows["input_tokens"] = NORMAL_INPUT * rows["count"]
        mark = rng.random(n)
        rows["result"] = np.choose(
            np.searchsorted(np.cumsum(NORMAL_ERROR), mark),
            (THROTTLED, VALIDATION, MISSING, OK),
        )
        for col in ("input_tokens", "output_tokens"):
            rows[col][rows["result"] != OK] = 0
        extra = []
        if key == HEALTH_KEY:
            extra.append(
                normal_calls(
                    np.arange(0, len(t), HEALTH_INTERVAL),
                    MID,
                    1,
                    ops_rng,
                )
            )
        if key == SMOKE_KEY:
            for start in deployments:
                seconds = burst_times(
                    ops_rng, start, SMOKE_SECONDS, SMOKE_CALLS
                )
                extra.append(
                    normal_calls(seconds, MID, 1, ops_rng)
                )
                if ops_rng.random() < TYPO_P:
                    seconds = burst_times(
                        ops_rng,
                        start,
                        SMOKE_SECONDS,
                        TYPO_CALLS,
                    )
                    typo = normal_calls(
                        seconds, HIGH + 1, CAPS[0], ops_rng
                    )
                    typo["result"] = MISSING
                    (
                        typo["input_tokens"],
                        typo["output_tokens"],
                    ) = 0, 0
                    extra.append(typo)
            admin = empty_rows(len(enables))
            admin["second"], admin["count"], admin["op"] = (
                enables,
                1,
                ENABLE,
            )
            extra.append(admin)
        if key < DEV_KEYS:
            candidates = [
                m
                for m in (LOW, MID, HIGH)
                if m != primary
                and not (key % 2 and m == 1 - primary)
            ]
            for week in range(0, len(t), WEEK):
                if (
                    candidates
                    and ops_rng.random() < MODEL_TRIAL_P
                ):
                    start = week + ops_rng.integers(
                        min(WEEK, len(t) - week)
                        - MODEL_TRIAL_SECONDS
                        + 1
                    )
                    seconds = burst_times(
                        ops_rng,
                        start,
                        MODEL_TRIAL_SECONDS,
                        MODEL_TRIAL_CALLS,
                    )
                    model = candidates.pop(
                        ops_rng.integers(len(candidates))
                    )
                    extra.append(
                        normal_calls(
                            seconds,
                            model,
                            ops_rng.choice(CAPS, len(seconds)),
                            ops_rng,
                        )
                    )
        rows = ordered(rows, *extra)
        rows["key"] = key
        period, probability = (
            (WEEK, DEV_ROTATE_P)
            if key < DEV_KEYS
            else (DAY, OPS_ROTATE_P)
        )
        changes = []
        networks = [DEV_NET if key < DEV_KEYS else OPS_NET]
        for begin in range(0, len(t), period):
            if ops_rng.random() < probability:
                changes.append(
                    begin
                    + ops_rng.integers(
                        min(period, len(t) - begin)
                    )
                )
                networks.append(next_network)
                next_network += 1
        rows["source"] = np.asarray(networks)[
            np.searchsorted(
                changes, rows["second"], side="right"
            )
        ]
        streams.append(rows)
    return streams + [empty_rows()]


@dataclass
class Attack:
    key: int
    start: int
    amp: int
    throttle: float
    rows: np.ndarray


def attack_scenario(seed, start_day, scenario):
    rng = np.random.default_rng(
        np.random.SeedSequence([seed, 1])
    )
    key = int(rng.integers(len(RATES)))
    start = int((start_day - 1) * DAY + rng.integers(DAY))
    n = rng.integers(PROBES[0], PROBES[1] + 1)
    spray_n = rng.integers(SPRAY[0], SPRAY[1] + 1)
    # Give each probe and error event its own second.
    offsets = (
        rng.choice(
            PROBE_SECONDS - 1, n + spray_n, replace=False
        )
        + 1
    )
    probe = empty_rows(n + spray_n + 1)
    probe["second"] = start + np.r_[0, offsets]
    probe["count"], probe["model"], probe["max_tokens"] = (
        1,
        LOW,
        1,
    )
    probe["op"][n] = LIST
    probe["max_tokens"][n] = 0
    probe["result"][n + 1 :] = MISSING
    probe["model"][n + 1 :] = rng.integers(
        HIGH + 1, len(MODELS), spray_n
    )
    probe["max_tokens"][n + 1 :] = CAPS[0]
    amp = start + PROBE_SECONDS + PAUSE
    parts = [probe]
    if scenario == "enable":
        key, amp, parts = len(RATES), start + 1, []
    if scenario != "base":
        admin = empty_rows(1)
        admin["second"], admin["count"] = amp - 1, 1
        admin["op"] = ENABLE if scenario == "enable" else DELETE
        parts.append(admin)
    elapsed = np.arange(AMP_SECONDS)
    counts = rng.poisson(
        PEAK * np.minimum((elapsed + 1) / RAMP, 1.0)
    )
    throttle = rng.uniform(*THROTTLE_RANGE)
    failed = rng.binomial(counts, throttle)
    for result, c in (
        (OK, counts - failed),
        (THROTTLED, failed),
    ):
        used = c > 0
        rows = empty_rows(used.sum())
        rows["second"], rows["count"] = (
            amp + elapsed[used],
            c[used],
        )
        rows["model"], rows["result"], rows["max_tokens"] = (
            HIGH,
            result,
            CAPS[-1],
        )
        parts.append(rows)
    rows = ordered(*parts)
    rows["source"], rows["key"] = ATTACK_NET, key
    return Attack(key, start, amp, float(throttle), rows)


def tokens(rows, profile):
    rows = rows.copy()
    billed = (rows["result"] == OK) & (rows["op"] == INVOKE)
    inp, out = PROFILES[profile]
    rows["input_tokens"][billed] = rows["count"][billed] * inp
    rows["output_tokens"][billed] = rows["count"][
        billed
    ] * np.minimum(out, rows["max_tokens"][billed])
    return rows


def charges(rows):
    """Bill only successful invocations."""
    billed = (rows["op"] == INVOKE) & (rows["result"] == OK)
    cost = np.zeros(len(rows), dtype="i8")
    rates = PRICE_UNITS[rows["model"][billed]]
    cost[billed] = (
        rows["input_tokens"][billed] * rates[:, 0]
        + rows["output_tokens"][billed] * rates[:, 1]
    )
    return cost

## 탐지 규칙과 정책

R1~R7은 소량 토큰 호출, 새 모델·대역, 호출량·요금 급증, 관리 이벤트, 오류를 감시합니다. R1b는 소량 토큰 호출의 비율을 과거와 비교하고 R3b는 새 대역 경보를 다른 신호와 대조합니다. P1~P6은 규칙을 조합하며 P4와 P6은 경보 다음 초부터 해당 키를 차단합니다.

In [ ]:
def empty_times():
    return np.empty(0, dtype="i4")


def by_second(times, values, length):
    out = np.zeros(length, dtype="i8")
    np.add.at(out, times, values)
    return out


class Latch:
    def __init__(self):
        self.latched = False

    def crossings(self, values, active, start):
        resets = np.cumsum(~active, dtype="i4")
        arrivals = np.flatnonzero((values > 0) & active)
        epochs = resets[arrivals]
        continued = self.latched and resets[-1] == 0
        previous = 0 if self.latched else -1
        self.latched = continued or bool(
            len(epochs)
            and epochs[-1] == resets[-1]
            and active[-1]
        )
        return (
            start
            + arrivals[np.diff(epochs, prepend=previous) != 0]
        ).astype("i4")


class Window(Latch):
    """Latch once per interval and retain width−1 seconds."""

    def __init__(self, width):
        self.width = width
        self.tail = np.empty(0, dtype="i8")
        super().__init__()

    def feed(self, values, threshold, start, strict=False):
        joined = np.concatenate((self.tail, values))
        total = np.cumsum(joined)
        total[self.width :] -= total[: -self.width]
        total = total[len(self.tail) :]
        self.tail = joined[-(self.width - 1) :].copy()
        active = (
            total > threshold if strict else total >= threshold
        )
        # Only a contributing arrival may emit. Count resets
        # even in silent seconds when observations expire.
        return self.crossings(values, active, start)


class ProbeRatio(Latch):
    """Compare count-weighted shares in two disjoint windows."""

    def __init__(self, minimum):
        super().__init__()
        self.minimum = minimum
        self.tail = np.empty((0, 2), dtype="i8")

    def feed(self, probes, calls, start):
        joined = np.vstack(
            (self.tail, np.column_stack((probes, calls)))
        )
        prefix = np.vstack(
            (
                np.zeros((1, 2), dtype="i8"),
                joined.cumsum(axis=0),
            )
        )
        end = len(self.tail) + np.arange(1, len(calls) + 1)
        split = np.maximum(end - PROBE_WINDOW, 0)
        recent_probes, recent_calls = (
            prefix[end] - prefix[split]
        ).T
        past_probes, past_calls = (
            prefix[split]
            - prefix[np.maximum(split - PROBE_HISTORY, 0)]
        ).T
        self.tail = joined[
            -(PROBE_HISTORY + PROBE_WINDOW) :
        ].copy()
        active = (
            (start + np.arange(len(calls)) >= LEARN)
            & (recent_probes >= self.minimum)
            & (past_calls > 0)
            & (
                recent_probes * past_calls
                > PROBE_RATIO_K * past_probes * recent_calls
            )
        )
        return self.crossings(probes, active, start)


def signal_pairs(
    networks, signals, last_signal=-CORRELATION_WINDOW
):
    for net in networks:
        index = signals.searchsorted(net, side="right")
        previous = signals[index - 1] if index else last_signal
        if 0 <= net - previous < CORRELATION_WINDOW:
            yield net, previous
        elif (
            index < len(signals)
            and signals[index] - net < CORRELATION_WINDOW
        ):
            yield net, signals[index]
        else:
            yield net, None


class Corroboration:
    """Emit once per new network when both signals are known."""

    def __init__(self):
        self.pending = empty_times()
        self.last_signal = -CORRELATION_WINDOW

    def feed(self, networks, signals, end):
        pending = []
        hits = []
        for net, signal in signal_pairs(
            np.r_[self.pending, networks],
            signals,
            self.last_signal,
        ):
            if signal is not None:
                hits.append(max(net, signal))
            elif end - net < CORRELATION_WINDOW:
                pending.append(net)
        self.pending = np.asarray(pending, dtype="i4")
        if len(signals):
            self.last_signal = signals[-1]
        return np.unique(hits).astype("i4")


class HourHistory:
    """Compute each hourly threshold before adding that hour."""

    def __init__(self, rate_k=None):
        self.rate_k = rate_k
        self.complete = []
        self.partial = np.empty(0, dtype="i8")

    def thresholds(self, values):
        offset = len(self.partial)
        joined = np.concatenate((self.partial, values))
        thresholds = []
        for i in range(0, len(joined), HOUR):
            h = len(self.complete)
            hour = joined[i : i + HOUR]
            threshold = np.inf
            if self.rate_k is None:
                if h >= BURN_MIN_HOURS:
                    past = self.complete[-BURN_HISTORY:]
                    threshold = BURN_K * np.quantile(
                        past, BURN_Q
                    )
            elif h >= 7 * 24:
                past = self.complete[h % (7 * 24) :: 7 * 24]
                threshold = max(
                    RATE_MIN, self.rate_k * np.median(past)
                )
            thresholds.append(threshold)
            if len(hour) == HOUR:
                observation = (
                    hour.sum()
                    if self.rate_k is None
                    else hour.reshape(-1, RATE_WINDOW).sum(
                        axis=1
                    )
                )
                self.complete.append(observation)
                self.partial = np.empty(0, dtype="i8")
            else:
                self.partial = hour.copy()
        return np.repeat(thresholds, HOUR)[offset : len(joined)]


class KeyState:
    def __init__(self, key, n=PROBE_N, k=RATE_K):
        self.second = 0
        self.n = n
        self.probes = Window(PROBE_WINDOW)
        self.probe_ratio = ProbeRatio(n)
        self.corroboration = Corroboration()
        self.errors = Window(ERROR_WINDOW)
        self.rate = Window(RATE_WINDOW)
        self.history = HourHistory(rate_k=k)
        self.models = {}
        self.networks = set()
        if PRELOAD_NETWORK and key < len(RATES):
            self.networks.add(
                DEV_NET if key < DEV_KEYS else OPS_NET
            )

    def feed(self, rows, end):
        """Read sorted key rows from [self.second, end)."""
        start = self.second
        size = end - start
        if size <= 0:
            raise ValueError("feed must advance the clock")
        invokes = rows[rows["op"] == INVOKE]
        ts, counts = invokes["second"], invokes["count"]
        offsets = ts - start
        rate = by_second(offsets, counts, size)
        small = invokes["max_tokens"] <= 1
        errors = np.isin(
            invokes["result"], (MISSING, VALIDATION)
        )
        probes = by_second(offsets[small], counts[small], size)
        failed = by_second(
            offsets[errors], counts[errors], size
        )
        alerts = {
            "R1": self.probes.feed(probes, self.n, start),
            "R1b": self.probe_ratio.feed(probes, rate, start),
            "R7": self.errors.feed(failed, ERROR_N, start),
        }
        new_models = []
        for model in np.unique(invokes["model"]):
            times = ts[invokes["model"] == model]
            prior = self.models.get(model, -MODEL_HISTORY - 1)
            fresh = (
                np.diff(times, prepend=prior) > MODEL_HISTORY
            )
            new_models.extend(times[fresh & (times >= LEARN)])
            self.models[model] = int(times[-1])
        alerts["R2"] = np.unique(new_models).astype("i4")
        new_networks = []
        for network, first in zip(
            *np.unique(invokes["source"], return_index=True)
        ):
            if network not in self.networks:
                if ts[first] >= LEARN:
                    new_networks.append(ts[first])
                self.networks.add(network)
        alerts["R3"] = np.unique(new_networks).astype("i4")
        signals = np.union1d(alerts["R1b"], alerts["R2"])
        alerts["R3b"] = self.corroboration.feed(
            alerts["R3"], signals, end
        )
        alerts["R4"] = self.rate.feed(
            rate, self.history.thresholds(rate), start
        )
        admin = np.isin(rows["op"], (ENABLE, DELETE))
        alerts["R6"] = np.unique(rows["second"][admin])
        self.second = end
        return alerts


class BurnState:
    def __init__(self):
        self.second = 0
        self.window = Window(BURN_WINDOW)
        self.history = HourHistory()

    def feed(self, spend):
        thresholds = self.history.thresholds(spend)
        alerts = self.window.feed(
            spend, thresholds, self.second, strict=True
        )
        self.second += len(spend)
        return alerts


def key_scan(rows, length, key, n=PROBE_N, k=RATE_K):
    state = KeyState(key, n, k)
    chunks = []
    for start in range(0, length, DAY):
        end = min(length, start + DAY)
        lo, hi = np.searchsorted(rows["second"], [start, end])
        chunks.append(state.feed(rows[lo:hi], end))
    return {
        rule: np.concatenate([chunk[rule] for chunk in chunks])
        for rule in chunks[0]
    }


def burn_scan(spend):
    state = BurnState()
    return np.concatenate(
        [
            state.feed(spend[i : i + DAY])
            for i in range(0, len(spend), DAY)
        ]
    )


def applicable(alerts, attack):
    """Match each alert to attack activity in its own window."""
    rows = attack.rows
    invoked = rows["op"] == INVOKE
    sources = {
        "R1": (
            invoked & (rows["max_tokens"] <= 1),
            PROBE_WINDOW,
        ),
        "R1b": (
            invoked & (rows["max_tokens"] <= 1),
            PROBE_WINDOW,
        ),
        "R2": (invoked, 1),
        "R3": (invoked, 1),
        "R4": (invoked, RATE_WINDOW),
        "R5": (invoked & (rows["result"] == OK), BURN_WINDOW),
        "R6": (np.isin(rows["op"], (ENABLE, DELETE)), 1),
        "R7": (
            np.isin(rows["result"], (VALIDATION, MISSING)),
            ERROR_WINDOW,
        ),
    }
    detected = {}
    for rule in sources:
        times = alerts[rule]
        mask, window = sources[rule]
        times = times[
            (times >= attack.start)
            & (times < attack.amp + AMP_SECONDS)
        ]
        events = rows["second"][mask]
        before = (
            np.searchsorted(events, times, side="right") - 1
        )
        times, before = times[before >= 0], before[before >= 0]
        hits = times[times - events[before] < window]
        detected[rule] = hits
    # At least one paired signal must come from the attack.
    signals = np.union1d(alerts["R1b"], alerts["R2"])
    related = np.union1d(detected["R1b"], detected["R2"])
    paired = []
    for network, signal in signal_pairs(alerts["R3"], signals):
        if signal is not None and (
            network in detected["R3"] or signal in related
        ):
            paired.append(max(network, signal))
    detected["R3b"] = np.intersect1d(alerts["R3b"], paired)
    end = attack.amp + AMP_SECONDS
    return {
        rule: next(
            (int(t) for t in times if attack.start <= t < end),
            None,
        )
        for rule, times in detected.items()
    }


def earliest(items):
    return min(
        (t for t in items if t is not None), default=None
    )


def add_spend(spend, ledger, cutoff, sign=1):
    times, costs = ledger
    stop = np.searchsorted(times, cutoff, side="right")
    np.add.at(spend, times[:stop], sign * costs[:stop])


def blocked_replay(local, ledgers, length, ruleset):
    """Replay organization spend using each key's first block.

    Advance the global clock after each additional block.
    """
    cuts = [
        min(
            (
                int(times[0])
                for rule, times in alerts.items()
                if rule in ruleset and len(times)
            ),
            default=length,
        )
        for alerts in local
    ]
    spend = np.zeros(length, dtype="i8")
    for key, ledger in ledgers:
        add_spend(spend, ledger, cuts[key])
    last = -1
    burn_blocks = {}
    while True:
        alarms = burn_scan(spend)
        for trigger in alarms[alarms > last]:
            t = int(trigger)
            amounts = np.zeros(len(cuts), dtype="i8")
            for key, (times, costs) in ledgers:
                if cuts[key] >= t:
                    a, b = np.searchsorted(times, [t, t + 1])
                    amounts[key] += costs[a:b].sum()
            key = int(amounts.argmax())
            burn_blocks[key] = t
            if cuts[key] == t:
                continue
            old, cuts[key] = cuts[key], t
            for owner, (times, costs) in ledgers:
                if owner == key:
                    a, b = np.searchsorted(
                        times, [t + 1, old + 1]
                    )
                    np.add.at(spend, times[a:b], -costs[a:b])
            last = t
            break
        else:
            return cuts, burn_blocks

## 실험과 결과 표

각 시드에서 정상 로그의 경보 수를 세고 세 공격 시나리오를 두 토큰 프로필로 비교합니다. 표에는 탐지율, 시작·증폭 기준 지연, 경보까지의 호출 수와 요금, 정상 조직·일당 경보가 표시됩니다.

In [ ]:
def record(t, attack, attack_times, cumulative, calls):
    if t is None:
        return dict(
            second=None,
            from_start_min=None,
            from_amp_min=None,
            usd=None,
            calls=None,
        )
    idx = np.searchsorted(attack_times, t, side="right")
    cost = int(cumulative[idx]) / MONEY_UNIT
    return dict(
        second=t,
        from_start_min=(t - attack.start) / 60,
        from_amp_min=(t - attack.amp) / 60,
        usd=cost,
        calls=int(calls[idx]),
    )


def normal_control(seed, days):
    length = days * DAY
    normal = normal_traffic(seed, days)
    local = [
        key_scan(rows, length, key)
        for key, rows in enumerate(normal)
    ]
    ledgers = [
        (key, (rows["second"], charges(rows)))
        for key, rows in enumerate(normal)
    ]
    spend = np.zeros(length, dtype="i8")
    for _, ledger in ledgers:
        add_spend(spend, ledger, length)
    burn = burn_scan(spend)
    per_key = [dict(alerts, R5=[]) for alerts in local]
    for t in burn:
        amounts = []
        for _, (times, costs) in ledgers:
            lo, hi = np.searchsorted(times, [t, t + 1])
            amounts.append(costs[lo:hi].sum())
        per_key[int(np.argmax(amounts))]["R5"].append(t)

    def daily(times):
        return np.bincount(
            np.asarray(times, dtype=int) // DAY, minlength=days
        ).tolist()

    false = {
        rule: daily(
            np.concatenate([key[rule] for key in per_key])
        )
        for rule in RULES
    }
    for policy, ruleset in POLICIES.items():
        if policy in BLOCKING:
            cuts, _ = blocked_replay(
                local, ledgers, length, ruleset
            )
            false[policy] = daily(
                [t for t in cuts if t < length]
            )
        else:
            emissions = [
                np.unique(
                    np.concatenate(
                        [key[rule] for rule in ruleset]
                    )
                )
                for key in per_key
            ]
            false[policy] = daily(np.concatenate(emissions))
    return normal, local, ledgers, spend, false


def run_seed(seed, days, start_day):
    length = days * DAY
    normal, local, ledgers, spend, false = normal_control(
        seed, days
    )
    experiments = {}
    timeline = None
    for scenario in SCENARIOS:
        attack = attack_scenario(seed, start_day, scenario)
        combined = ordered(normal[attack.key], attack.rows)
        case_local = local.copy()
        case_local[attack.key] = key_scan(
            combined, length, attack.key
        )
        profile_results = {}
        for profile in PROFILES:
            billed = tokens(attack.rows, profile)
            times = billed["second"]
            cost = charges(billed)
            incident_spend = spend.copy()
            add_spend(incident_spend, (times, cost), length)
            alerts = {
                **case_local[attack.key],
                "R5": burn_scan(incident_spend),
            }
            first = applicable(alerts, attack)
            cumulative = np.r_[np.int64(0), np.cumsum(cost)]
            calls = np.r_[
                np.int64(0),
                np.cumsum(
                    np.where(
                        billed["op"] == INVOKE,
                        billed["count"],
                        0,
                    )
                ),
            ]
            blocks = {}
            for policy, ruleset in POLICIES.items():
                if policy not in BLOCKING:
                    first[policy] = earliest(
                        first[rule] for rule in ruleset
                    )
                    continue
                cuts, burn_blocks = blocked_replay(
                    case_local,
                    ledgers + [(attack.key, (times, cost))],
                    length,
                    ruleset,
                )
                cut = cuts[attack.key]
                attributable = applicable(
                    {
                        **alerts,
                        "R5": np.array(
                            [
                                burn_blocks.get(
                                    attack.key, length
                                )
                            ]
                        ),
                    },
                    attack,
                )
                first[policy] = (
                    cut
                    if any(
                        attributable[rule] == cut
                        for rule in ruleset
                    )
                    else None
                )
                blocks[policy] = dict(
                    blocked_usd=record(
                        cut, attack, times, cumulative, calls
                    )["usd"],
                    block_second=cut if cut < length else None,
                    blocked_before_attack=cut < attack.start,
                )
            measures = {
                m: record(
                    first[m], attack, times, cumulative, calls
                )
                for m in METRICS
            }
            profile_results[profile] = dict(
                metrics=measures,
                unblocked_usd=int(cumulative[-1]) / MONEY_UNIT,
                blocks=blocks,
                **blocks["P4"],
            )
            if (
                seed == 0
                and scenario == "base"
                and profile == "long"
            ):
                calls = combined[combined["op"] == INVOKE]
                per_min = by_second(
                    calls["second"] // 60,
                    calls["count"],
                    days * 1440,
                )
                timeline = (
                    per_min,
                    attack.start,
                    {
                        rule: first[rule]
                        for rule in ORIGINAL_RULES
                    },
                )
        experiments[scenario] = dict(
            key=KEYS[attack.key],
            start=attack.start,
            amp=attack.amp,
            throttle=attack.throttle,
            profiles=profile_results,
        )
    return dict(
        seed=seed,
        normal_daily_alerts=false,
        normal_alert_counts={
            k: sum(v) for k, v in false.items()
        },
        scenarios=experiments,
    ), timeline


def false_summary(daily_runs):
    result = {}
    for name in METRICS:
        observations = np.concatenate(
            [run[name] for run in daily_runs]
        )
        result[name] = dict(
            alerts=int(observations.sum()),
            days=len(observations),
            alert_days=int(np.count_nonzero(observations)),
            per_day=float(observations.mean()),
            day_rate=float((observations > 0).mean()),
        )
    return result


def summary(raw, days):
    result = {}
    for scenario in SCENARIOS:
        result[scenario] = {}
        for profile in PROFILES:
            cases = [
                run["scenarios"][scenario]["profiles"][profile]
                for run in raw
            ]
            values = {}
            for name in METRICS:
                hits = [
                    case["metrics"][name]
                    for case in cases
                    if case["metrics"][name]["second"]
                    is not None
                ]

                def stats(field):
                    data = [h[field] for h in hits]
                    return (
                        [
                            float(f(data))
                            for f in (np.median, np.min, np.max)
                        ]
                        if data
                        else [None] * 3
                    )

                values[name] = dict(
                    hits=len(hits),
                    trials=len(cases),
                    rate=len(hits) / len(cases),
                    start=stats("from_start_min"),
                    amp=stats("from_amp_min"),
                    usd=stats("usd"),
                    calls=stats("calls"),
                )
            result[scenario][profile] = dict(
                metrics=values,
                preblocks=sum(
                    c["blocked_before_attack"] for c in cases
                ),
                blocked=float(
                    np.median([c["blocked_usd"] for c in cases])
                ),
                blocks={
                    p: dict(
                        preblocks=sum(
                            c["blocks"][p][
                                "blocked_before_attack"
                            ]
                            for c in cases
                        ),
                        usd=float(
                            np.median(
                                [
                                    c["blocks"][p][
                                        "blocked_usd"
                                    ]
                                    for c in cases
                                ]
                            )
                        ),
                    )
                    for p in BLOCKING
                },
                unblocked=float(
                    np.median(
                        [c["unblocked_usd"] for c in cases]
                    )
                ),
            )
    return result, false_summary(
        [run["normal_daily_alerts"] for run in raw]
    )


def fmt(x):
    return "—" if x is None else f"{x:.5g}"


def tables(summaries, false, fx):
    for scenario, profs in summaries.items():
        for profile, case in profs.items():
            line()
            line(
                f"[{scenario} / {profile}] 지연(분) = 중앙값 [최소,최대]"
            )
            line(
                "항목 탐지율 경보 전 공격 호출(중앙) 시작/증폭 기준 분"
            )
            for name, m in case["metrics"].items():

                def interval(triple):
                    v, lo, hi = triple
                    return (
                        "—"
                        if v is None
                        else f"{v:.2f} [{lo:.2f},{hi:.2f}]"
                    )

                line(
                    f"{name:3} {m['rate']:4.0%} "
                    f"{fmt(m['calls'][0]):>6}  "
                    f"{interval(m['start'])}  "
                    f"{interval(m['amp'])}"
                )
            line(
                "항목   경보 전 공격 호출(중앙)   공격 요금 중앙 (USD / KRW)"
            )
            for name, m in case["metrics"].items():
                usd = m["usd"][0]
                krw = usd * fx if usd is not None else None
                line(
                    f"{name:3} {fmt(m['calls'][0]):>8}    "
                    f"{fmt(usd)} / {fmt(krw)}"
                )
            for policy, block in case["blocks"].items():
                usd = block["usd"]
                line(
                    f"{policy} 사전 차단 {block['preblocks']}건 / "
                    f"총 공격 요금 중앙 {usd:,.5f} USD / "
                    f"{usd * fx:,.2f}원"
                )
            usd = case["unblocked"]
            line(
                f"미차단 총 공격 요금 중앙 {usd:,.5f} USD / "
                f"{usd * fx:,.2f}원"
            )
    line()
    line("정상 로그 / 발생 조직·일은 경보가 1건 이상 있었던 날")
    line(
        "항목  경보 전 공격 호출    경보/일  발생 조직·일/전체    비율"
    )
    for name, m in false.items():
        days = f"{m['alert_days']}/{m['days']}"
        line(
            f"{name:4}  {0:17}  {m['per_day']:9.6f}  "
            f"{days:>17}  {m['day_rate']:6.2%}"
        )


begin = time.perf_counter()
raw = []
for seed in range(trials):
    item, example = run_seed(seed, days, start)
    raw.append(item)
    if seed == 0:
        timeline = example
        per_seed = time.perf_counter() - begin
        line(
            f"첫 시드 기준 현재 실험 예상 "
            f"{per_seed * trials:.1f}초 (그래프 제외)"
        )
runtime = time.perf_counter() - begin
# Quick has no second-week R4 baseline.
estimated = (
    runtime
    / trials
    * SEEDS
    * (DAYS / days)
    * (2 if QUICK else 1)
)
line(
    f"시뮬레이션 실측 {runtime:.2f}초 / "
    f"전체 예상 {estimated:.1f}초"
)
if QUICK:
    line(
        "같은 CPU 기준 / 시드·일수 선형 환산 ×2 (2주차 R4 여유)"
    )
summaries, false = summary(raw, days)
tables(summaries, false, fx)

## 그래프

공격 타임라인, 정책별 경보 시점 요금, 규칙별 정상 조직·일당 경보 수를 `results/`에 PNG로 저장하고 아래에 표시합니다. `both`는 각 파일명에 `_ko` 또는 `_en`을 붙입니다. 한글 그래프에는 실행 시 내려받는 NanumGothic을 사용합니다. [Google Fonts · NanumGothic, SIL Open Font License 1.1](https://github.com/google/fonts/tree/main/ofl/nanumgothic)

In [ ]:
from IPython.display import Image, display

FONT_URL = (
    "https://github.com/google/fonts/raw/main/"
    "ofl/nanumgothic/NanumGothic-Regular.ttf"
)


def register_font(fm):
    font = Path("cache/fonts/NanumGothic-Regular.ttf")
    try:
        if not font.exists():
            with urlopen(FONT_URL, timeout=5) as response:
                data = response.read()
            font.parent.mkdir(parents=True, exist_ok=True)
            font.write_bytes(data)
        fm.fontManager.addfont(font)
        name = fm.FontProperties(fname=font).get_name()
        fm.findfont(
            fm.FontProperties(family=name),
            fallback_to_default=False,
        )
        return name
    except Exception as exc:
        line(
            f"경고 NanumGothic 등록 실패 ({type(exc).__name__})"
        )
        available = {f.name for f in fm.fontManager.ttflist}
        for name in (
            "NanumGothic",
            "Apple SD Gothic Neo",
            "Malgun Gothic",
            "Noto Sans CJK KR",
            "AppleGothic",
        ):
            if name in available:
                line(f"한글 대체 글꼴 {name}")
                return name
        line("한글 대체 글꼴 없음 / 일부 글자가 누락될 수 있음")
        return "DejaVu Sans"


def charts(directory, lang, timeline, summaries, false):
    import matplotlib

    matplotlib.use("Agg")
    from matplotlib import pyplot as plt, font_manager, ticker

    family = (
        register_font(font_manager)
        if lang != "en"
        else "DejaVu Sans"
    )
    minutes, start, first = timeline
    ink, accent, gray = "#426273", "#a26850", "#768c81"
    base = summaries["base"]["long"]["metrics"]
    paths = []
    for language in ("ko", "en") if lang == "both" else (lang,):
        ko = language == "ko"
        suffix = "_" + language if lang == "both" else ""

        def save(fig, name):
            path = directory / f"{name}{suffix}.png"
            fig.savefig(path, dpi=150)
            paths.append(path)
            plt.close(fig)

        with plt.rc_context(
            {
                "font.family": family if ko else "DejaVu Sans",
                "axes.unicode_minus": False,
                "axes.spines.top": False,
                "axes.spines.right": False,
                "font.size": 10,
            }
        ):

            def axes():
                return plt.subplots(
                    figsize=(8, 4.2), layout="constrained"
                )

            fig, ax = axes()
            x = (np.arange(len(minutes)) * 60 - start) / 3600
            shown = (x >= -2) & (
                x
                <= (PROBE_SECONDS + PAUSE + AMP_SECONDS) / 3600
                + 1
            )
            ax.plot(
                x[shown],
                np.ma.masked_equal(minutes[shown], 0),
                color=ink,
                lw=1,
                label="호출" if ko else "Calls",
            )
            ax.set_yscale("log")
            # Stagger labels when alarms land together.
            first_line = True
            for i, (rule, sec) in enumerate(first.items()):
                if sec is None:
                    continue
                loc = (sec - start) / 3600
                ax.axvline(
                    loc,
                    color=accent,
                    ls="--",
                    lw=1,
                    alpha=0.85,
                    label=("첫 경보" if ko else "First alert")
                    if first_line
                    else None,
                )
                first_line = False
                ax.annotate(
                    rule,
                    (loc, 0.92 - i * 0.07),
                    xycoords=("data", "axes fraction"),
                    xytext=(3, 0),
                    textcoords="offset points",
                    color=accent,
                )
            missing = ", ".join(
                rule
                for rule, sec in first.items()
                if sec is None
            )
            if missing:
                ax.text(
                    0.99,
                    0.32,
                    ("미탐 " if ko else "No alert  ") + missing,
                    ha="right",
                    transform=ax.transAxes,
                )
            ax.legend(loc="upper right")
            ax.set(
                xlabel="공격 시작 이후 시간 (h)"
                if ko
                else "Hours from attack start",
                ylabel="공격 키 호출 (건/분)"
                if ko
                else "Calls per minute on exposed key",
            )
            ax.grid(axis="y", alpha=0.18)
            save(fig, "fig1_attack_timeline")
            fig, ax = axes()
            names = tuple(POLICIES)
            amounts = [base[p]["usd"][0] for p in names]
            positive = [
                v for v in amounts if v is not None and v > 0
            ]
            left = (
                10.0
                ** (math.floor(math.log10(min(positive))) - 1)
                if positive
                else 0.0001
            )
            right = max(positive, default=1.0) * 3
            ax.set_xscale("log")
            ax.set_xlim(left, right)
            ax.set_yticks(range(len(names)), names)
            ax.set_ylim(-0.55, len(names) - 0.45)
            ax.xaxis.set_major_formatter(
                ticker.FuncFormatter(lambda x, _: f"{x:g}")
            )
            for i, v in enumerate(amounts):
                if v is not None and v > 0:
                    ax.barh(
                        i,
                        v - left,
                        left=left,
                        color=ink,
                        height=0.55,
                    )
                else:
                    label = (
                        ("미탐" if ko else "No alert")
                        if v is None
                        else "0 USD"
                    )
                    ax.text(
                        0.015,
                        i,
                        label,
                        va="center",
                        transform=ax.get_yaxis_transform(),
                    )
            ax.invert_yaxis()
            ax.set_xlabel(
                "기본 공격 · 긴 컨텍스트 · 경보까지 중앙 요금 (USD)"
                if ko
                else "Base attack, long context: "
                "median cost at alert (USD)"
            )
            ax.set_ylabel("정책" if ko else "Policy")
            ax.grid(axis="x", alpha=0.18)
            save(fig, "fig2_cost_before_alert")
            fig, ax = axes()
            rates = [false[rule]["per_day"] for rule in RULES]
            bars = ax.bar(RULES, rates, color=gray, width=0.6)
            ax.bar_label(
                bars,
                labels=[f"{v:.4g}" for v in rates],
                padding=3,
            )
            ax.set_ylim(0, max(0.01, max(rates) * 1.2))
            ax.set(
                xlabel="규칙" if ko else "Rule",
                ylabel="정상 조직 하루 평균 경보 (건)"
                if ko
                else "Mean false alarms per organization-day",
            )
            ax.grid(axis="y", alpha=0.18)
            save(fig, "fig3_false_alarms")
    return paths


directory = Path("results")
directory.mkdir(exist_ok=True)
with tempfile.TemporaryDirectory() as cache:
    os.environ.setdefault("MPLCONFIGDIR", cache)
    chart_paths = charts(
        directory, CHART_LANG, timeline, summaries, false
    )
for path in chart_paths:
    display(Image(filename=str(path)))

## 결과 저장

실행 가정, 시드별 지표, 오탐 집계, 타임라인을 `results/results.json`에 저장합니다. Colab 왼쪽의 파일 탭에서 JSON과 PNG를 내려받을 수 있습니다.

In [ ]:
for seed in raw:
    for case in seed["scenarios"].values():
        for p in case["profiles"].values():
            for metric in p["metrics"].values():
                metric["krw"] = (
                    None
                    if metric["usd"] is None
                    else metric["usd"] * fx
                )
            for k in ("blocked", "unblocked"):
                p[k + "_krw"] = p[k + "_usd"] * fx
            for block in p["blocks"].values():
                block["blocked_krw"] = block["blocked_usd"] * fx
payload = dict(
    version=VERSION,
    utc=utc,
    quick=QUICK,
    days=days,
    seeds=trials,
    krw_per_usd=fx,
    assumptions=setup,
    runs=raw,
    summary=summaries,
    false_alerts=false,
    false_per_day={k: v["per_day"] for k, v in false.items()},
    simulation_seconds=runtime,
    estimated_full_seconds=estimated,
    timeline_seed0=dict(
        minute_counts=timeline[0].tolist(),
        start=timeline[1],
        first_long=timeline[2],
    ),
)
(directory / "results.json").write_text(
    json.dumps(
        payload, ensure_ascii=False, allow_nan=False, indent=2
    )
    + "\n"
)

line("저장 완료 results/results.json 및 PNG")